# cuOpt Agent

The cuOpt Agent is a reference optimization assistant that translates natural-language problem descriptions into mathematical models, solves them on the GPU with NVIDIA cuOpt, and returns explained results. It ships with a multi-period supply chain planning scenario (max-supply) as its built-in use case, but the architecture is designed to be extended to other LP/MILP domains by adding new skills, data files, and configs. It is built as a LangChain Deep Agents workflow on top of NAT, with structured skills that make the path from problem text to correct formulation more reliable. The agent uses an orchestrator-subagent pattern: the orchestrator receives user queries and delegates optimization work to a specialized cuOpt subagent via the task() tool, keeping coordination logic separate from problem-solving logic.

Key features:

- GPU-accelerated solver: cuOpt LP/MILP solver for fast optimization on NVIDIA GPUs.
- Structured skills: Parsing, modeling, debugging, and supply-chain planning skills give the agent rules and reference models so it formulates correctly on the first try.
- Mandatory ambiguity handling: When problem text is ambiguous, the agent must clarify or solve all plausible interpretations.
- YAML-driven configuration: Agents, LLMs, and workflows are defined in config files; tune behavior without code changes.
- Web UI: Chat interface for interacting with the agent.
- Tracing: Phoenix and LangSmith (optional) for inspecting agent behavior.
- Evaluation harness: Built-in eval configs for measuring agent quality.

### Software Components

| Component | Purpose |
|-----------|---------|
| [NVIDIA NeMo Agent Toolkit](https://docs.nvidia.com/nemo/agent-toolkit/latest/) | Agent framework and serving |
| [NVIDIA cuOpt](https://developer.nvidia.com/cuopt) | GPU-accelerated LP/MILP solver |
| [LangChain Deep Agents](https://www.langchain.com/) | Multi-step agent workflow |
| [minimaxai/minimax-m2.5](https://build.nvidia.com/) (via NIM) | LLM for agent reasoning |
| [Phoenix](https://github.com/Arize-ai/phoenix) | OpenTelemetry tracing |
| [LangSmith](https://smith.langchain.com/) (optional) | Agent tracing and observability |
| [NAT UI](external/nat-ui/) | Chat web interface |


### Prerequisites

- NVIDIA GPU with CUDA support
- Docker and Docker Compose (with [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html))
- NVIDIA API key from [build.nvidia.com](https://build.nvidia.com/)

Since you are running this in a Brev launchable environment, CUDA and Docker prereqs are taken care of. Before running through this notebook, ensure you have access to the cuOpt image and that you have an API key. 

**Optional:**

- [LangSmith](https://smith.langchain.com/) API key for agent tracing (uncomment the LangSmith section in the config YAML to enable)


**For local development (without Docker) and improved performance**
Check the guide [here](https://github.com/NVIDIA/cuopt-examples/tree/main/cuopt-agent#readme)

## 1. Clone the Repository
First, we will clone the cuOpt examples repostory, which has sample notebooks for a wide variety of use cases. The subdirectory `cuopt-agent` includes all the code needed for this demo. 
Let's clone the repo and move into the relevant subdirectory.

In [ ]:
!git clone https://github.com/NVIDIA/cuopt-examples.git

In [ ]:
%cd cuopt-examples/cuopt-agent

## 2. Initialize Prereqs
Let's initialize a couple submodules and environment variables.

In [ ]:
# paste here your API key from build.nvidia.com
import os
os.environ['NVIDIA_API_KEY'] = 'nvapi-***'

In [ ]:
# 1. Initialize submodules (required for the Chat UI)
! git submodule update --init --recursive

# 2. Set up environment variables
!cp .env.example .env

# save the API Key to the .env file
with open(".env", "a") as f:
    f.write(f"\nNVIDIA_API_KEY={os.environ['NVIDIA_API_KEY']}\n")



In [ ]:
# log in to NVIDIA container registry so we can access the cuOpt container
! docker login nvcr.io --username '$oauthtoken' --password {os.environ['NVIDIA_API_KEY']}

## 3. Running MiniMax NIM Locally

Open a new terminal tab and follow the instructions below. These instructions are taken from this [deployment guide](https://build.nvidia.com/minimaxai/minimax-m2.5/deploy?nim=self-hosted) with a few modifications.

The steps are:

1. Export your API key. The NGC_API_KEY is the same as the NVIDIA_API_KEY we set above. Just set it as a new environment variable with the specified name (NGC_API_KEY). You do not need to generate a new API key. 
```
export NGC_API_KEY=nvapi-***
```

2. Run the following commands/
```
export LOCAL_NIM_CACHE=~/.cache/nim
mkdir -p "$LOCAL_NIM_CACHE"
chmod -R a+w "$LOCAL_NIM_CACHE"
```

3. Run the NIM container. When you run the `docker run` command, modify it so the NIM uses port 8001 instead of 8000. Port 8000 will be used for the cuOpt Agent app in the next section. This should take around 10 minutes to run.
The new command is:
```
docker run -it --rm \
--gpus all \
--shm-size=16GB \
-e NGC_API_KEY \
-v "$LOCAL_NIM_CACHE:/opt/nim/.cache" \
-p 8001:8000 \
nvcr.io/nim/minimax-ai/minimax-m25:latest
```

4. Test that the NIM is working properly. Once the container finished building and is running, open a new terminal tab and make the request below. This is just a sample request. You should see a response from the LLM. This is the same test API call from the guide on build.nvidia.com. The only change is the port number, as we set it to 8001 instead of the default 8000.
```
curl -X 'POST' \
'http://0.0.0.0:8001/v1/chat/completions' \
-H 'accept: application/json' \
-H 'Content-Type: application/json' \
-d '{
    "model": "MiniMaxAI/MiniMax-M2.5",
    "messages": [{"role":"user", "content":"Which number is larger, 9.11 or 9.8?"}],
    "max_tokens": 64
}'
```

5. Lastly, let's update our config file to point to our local NIM endpoint. Navigate to the config file located in `cuopt-examples/cuopt_agent/configs/config-deepagent.yml` and update the llm section to the following config.

```
llms:
  cuopt_llm:
    _type: nim
    model: MiniMaxAI/MiniMax-M2.5
    api_key: ${NVIDIA_API_KEY}
    base_url: http://172.17.0.1:8001/v1
    max_tokens: 16384
    do_auto_retry: false
```

## 4. Build and start all services

We are ready to build and run the demo. Note that each one of the following commands may take a couple minutes to run. 

In [ ]:
! docker compose -f deploy/compose/docker-compose.yml build

In [ ]:
! docker compose -f deploy/compose/docker-compose.yml up -d

## 5. Open the Chat UI and Phoenix Tracing. 

Go back to the Brev Launchable page and go to the `Access` tab.

In the `Using Secure Links` section, you should the following services:
1) Jupyter running on port 8888. This is where we are now.
2) The Chat UI running on port 3000.
3) Phoenix tracing running on port 6006.

Open the chat UI and the phoenix tracing pages by clicking on the Shareable URLs. You are ready to start using the Agent!
   
There are sample prompts you can copy and paste in the UI in the [cuopt-examples repo](https://github.com/NVIDIA/cuopt-examples/tree/main/cuopt-agent/cuopt_agent/data/max_supply_what_ifs/sample_prompts).

In the UI, you can see the Agent planning and executing tasks, and the final output once it's done running.

